# LeetCode 1902 — Depth of BST Given Insertion Order bằng Python

Link đề: https://leetcode.com/problems/depth-of-bst-given-insertion-order/ · Độ khó: **Medium** · Chủ đề: Array, Tree, Binary Search Tree, Binary Tree, Ordered Set

> **Bài Premium.** Đề bài trong notebook được viết lại theo hiểu biết về bài (nguồn: bản mô tả của doocs/leetcode). Nếu có tài khoản Premium, hãy đối chiếu lại đề gốc trên LeetCode.

Skeleton LeetCode:

```python
class Solution:
    def maxDepthBST(self, order: List[int]) -> int:
        ...
```

Lộ trình:
1. **Chèn thật vào BST** (có lớp `TreeNode`), đi từ gốc xuống cho mỗi phần tử — `O(n²)` worst khi thứ tự tăng dần.
2. **Tập có thứ tự (SortedList)**: khi chèn `x`, nó trở thành con của **tiền nhiệm** hoặc **hậu bối** của nó trong tập đã chèn ⇒ `depth(x) = 1 + max(depth(pred), depth(succ))` — `O(n log n)`.
3. **Ngoại tuyến, danh sách liên kết đôi trên giá trị `1..n`**: xóa các giá trị theo thứ tự **ngược** để tìm tiền nhiệm/hậu bối tại thời điểm chèn — `O(n)`.
4. **Cây Cartesian**: BST theo khóa với "độ ưu tiên" = thời điểm chèn chính là một **treap**; dựng bằng ngăn xếp đơn điệu trên mảng `time[value]` — `O(n)`.

## 1. Mục tiêu học

1. Hiểu quá trình **chèn vào BST không cân bằng** và vì sao thứ tự chèn quyết định độ sâu (thứ tự tăng dần ⇒ cây suy biến).
2. Chứng minh bổ đề then chốt: nút mới `x` được gắn làm con của **tiền nhiệm hoặc hậu bối** (trong số các giá trị đã chèn) — cái nào **sâu hơn**.
3. Dùng **tập có thứ tự** (`SortedList`) để tìm tiền nhiệm/hậu bối trong `O(log n)`.
4. Biết kỹ thuật **xử lý ngoại tuyến bằng xóa ngược** trên danh sách liên kết đôi để có `O(n)`.
5. Nhận ra BST theo thứ tự chèn ≡ **cây Cartesian / treap** (khóa = giá trị, ưu tiên = thời điểm chèn) và dựng bằng **ngăn xếp đơn điệu**.

## 2. Đề bài & ví dụ

Cho mảng `order` độ dài `n` (chỉ số từ 0) là một **hoán vị** của `1..n`, biểu diễn **thứ tự chèn** vào một cây nhị phân tìm kiếm (BST):
- Cây con trái của một nút chỉ chứa khóa **nhỏ hơn**; cây con phải chỉ chứa khóa **lớn hơn**; hai cây con cũng là BST.
- `order[0]` là gốc; mỗi phần tử tiếp theo được chèn làm **con** của một nút hiện có sao cho giữ tính chất BST (chèn chuẩn: đi từ gốc xuống).

Trả về **độ sâu** của BST — số **nút** trên đường dài nhất từ gốc tới lá xa nhất.

**Ví dụ 1:** `order = [2,1,4,3]` → `3` (đường `2 → 4 → 3`).

**Ví dụ 2:** `order = [2,1,3,4]` → `3` (đường `2 → 3 → 4`).

**Ví dụ 3:** `order = [1,2,3,4]` → `4` (đường `1 → 2 → 3 → 4`).

**Ràng buộc:** `1 ≤ n ≤ 10⁵`, `order` là hoán vị của `1..n`.

(Ghi chú: bản doocs viết đường ví dụ 1 là `2 → 3 → 4`; thực tế chèn `[2,1,4,3]` cho `4` là con phải của `2` và `3` là con trái của `4`, nên đường dài nhất là `2 → 4 → 3` — độ sâu vẫn là `3`.)

In [1]:
import random
import time
from collections import deque
from typing import List, Optional

examples = [([2, 1, 4, 3], 3), ([2, 1, 3, 4], 3), ([1, 2, 3, 4], 4)]
for order, want in examples:
    print(f"order={order} -> kỳ vọng {want}")

order=[2, 1, 4, 3] -> kỳ vọng 3
order=[2, 1, 3, 4] -> kỳ vọng 3
order=[1, 2, 3, 4] -> kỳ vọng 4


## 3. Kiến thức nền

### 3.1 Lớp `TreeNode`, chèn BST lặp và độ sâu

In [2]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def insert(root, x):
    """Chèn x vào BST (lặp, không đệ quy). Trả về (root, độ sâu của nút mới)."""
    node = TreeNode(x)
    if root is None:
        return node, 1
    cur, d = root, 1
    while True:
        d += 1
        if x < cur.val:
            if cur.left is None:
                cur.left = node; return root, d
            cur = cur.left
        else:
            if cur.right is None:
                cur.right = node; return root, d
            cur = cur.right

def depth_iter(root):
    if root is None: return 0
    best, st = 0, [(root, 1)]
    while st:
        nd, d = st.pop(); best = max(best, d)
        if nd.left: st.append((nd.left, d + 1))
        if nd.right: st.append((nd.right, d + 1))
    return best

def to_level_list(root):
    out, q = [], deque([root])
    while q:
        nd = q.popleft()
        if nd is None: out.append(None); continue
        out.append(nd.val); q.append(nd.left); q.append(nd.right)
    while out and out[-1] is None: out.pop()
    return out

root = None
for x in [2, 1, 4, 3]:
    root, d = insert(root, x)
    print(f"chèn {x}: độ sâu nút mới = {d}")
print("cây (level-order):", to_level_list(root), "| độ sâu:", depth_iter(root))

chèn 2: độ sâu nút mới = 1
chèn 1: độ sâu nút mới = 2
chèn 4: độ sâu nút mới = 2
chèn 3: độ sâu nút mới = 3
cây (level-order): [2, 1, 4, None, None, 3] | độ sâu: 3


### 3.2 Tiền nhiệm và hậu bối trong tập đã sắp

Trong tập `S` đã chèn, **tiền nhiệm** của `x` là phần tử lớn nhất `< x`, **hậu bối** là phần tử nhỏ nhất `> x`. `bisect` trên list đã sắp tìm được cả hai.

In [3]:
from bisect import bisect_left, insort
S = [1, 2, 4]
x = 3
i = bisect_left(S, x)
print("tiền nhiệm:", S[i - 1] if i > 0 else None, "| hậu bối:", S[i] if i < len(S) else None)

tiền nhiệm: 2 | hậu bối: 4


### 3.3 `sortedcontainers.SortedList`

Cấu trúc tập có thứ tự (có sẵn trên LeetCode Python). `add` và `bisect_left` đều `O(log n)` (khấu hao).

In [4]:
from sortedcontainers import SortedList
sl = SortedList([5, 1, 3])
sl.add(4)
print(sl, sl.bisect_left(4), sl[sl.bisect_left(4) - 1])

SortedList([1, 3, 4, 5]) 2 3


### 3.4 Danh sách liên kết đôi trên mảng

`prv[v]`, `nxt[v]` cho giá trị `v`; xóa `v` bằng cách nối `prv[v]` với `nxt[v]` trong `O(1)`.

In [5]:
n = 5
prv = list(range(-1, n + 1))          # prv[v] = v - 1   (0 là lính canh trái)
nxt = list(range(1, n + 3))           # nxt[v] = v + 1   (n + 1 là lính canh phải)
def remove(v):
    nxt[prv[v]] = nxt[v]; prv[nxt[v]] = prv[v]
remove(3)
print("sau khi xóa 3: hàng xóm của 4 =", prv[4], nxt[4], "| hàng xóm của 2 =", prv[2], nxt[2])

sau khi xóa 3: hàng xóm của 4 = 2 5 | hàng xóm của 2 = 1 4


## 4. Ý tưởng nền tảng / trực giác

Chèn `order = [2, 1, 4, 3]`:

| bước | chèn | tập đã có | tiền nhiệm | hậu bối | gắn vào | độ sâu |
|---|---|---|---|---|---|---|
| 1 | 2 | {} | — | — | gốc | 1 |
| 2 | 1 | {2} | — | 2 | con trái của 2 | 2 |
| 3 | 4 | {1,2} | 2 | — | con phải của 2 | 2 |
| 4 | 3 | {1,2,4} | 2 (sâu 1) | 4 (sâu 2) | con trái của **4** | 3 |

**Quan sát:** nút mới luôn gắn vào tiền nhiệm hoặc hậu bối — và là cái **sâu hơn** (tiền nhiệm và hậu bối luôn có quan hệ tổ tiên–con cháu, nút mới gắn vào nút **sâu** trong hai nút đó). ⇒ `depth(x) = 1 + max(depth(pred), depth(succ))` (thiếu bên nào thì coi là 0).

## 5. Cách 1 — Brute Force / baseline: chèn thật vào BST

In [6]:
def max_depth_insert(order):
    root = None
    best = 0
    for x in order:
        root, d = insert(root, x)
        best = max(best, d)
    return best

for order, want in examples:
    print(max_depth_insert(order), "kỳ vọng", want)

3 kỳ vọng 3
3 kỳ vọng 3
4 kỳ vọng 4


**Độ phức tạp:** mỗi lần chèn đi xuống tới độ sâu hiện tại ⇒ `O(n · h)`; worst (thứ tự tăng dần) `h = n` ⇒ `O(n²) = 10¹⁰` với `n = 10⁵`. Space `O(n)`.

**Nút thắt:** đi từ gốc xuống mỗi lần. Ta chỉ cần biết **cha** của nút mới — và cha được xác định bởi tiền nhiệm/hậu bối.

In [7]:
for n in [2000, 5000]:
    t0 = time.perf_counter(); r = max_depth_insert(list(range(1, n + 1)))
    print(f"n={n} (tăng dần): độ sâu {r}, {time.perf_counter() - t0:.3f}s")

n=2000 (tăng dần): độ sâu 2000, 0.059s


n=5000 (tăng dần): độ sâu 5000, 0.368s


## 6. Ý tưởng thuật toán / bước đột phá

**Bổ đề.** Khi chèn `x` vào BST chứa tập `S`, gọi `p` = tiền nhiệm, `s` = hậu bối của `x` trong `S`. Khi đó `x` trở thành **con phải của `p`** hoặc **con trái của `s`**, và đó là nút **sâu hơn** trong hai nút.

*Lý do:* trong BST, `p` và `s` (hai phần tử kề nhau trong thứ tự sắp xếp, nếu cả hai tồn tại) thì một nút là tổ tiên của nút kia (không thể ở hai nhánh khác nhau của một nút `r`, vì khi đó `p < r < s`, mâu thuẫn với `p, s` kề nhau). Nếu `s` là tổ tiên của `p` thì `p` nằm trong cây con trái của `s`, và `p` là phần tử lớn nhất trong đó ⇒ `p.right` trống ⇒ `x` gắn vào `p.right` (sâu hơn). Đối xứng khi `p` là tổ tiên của `s`.

**Hệ quả:** `depth(x) = 1 + max(depth(p), depth(s))`.

**Ngoại tuyến:** nếu xét các giá trị `1..n` theo thứ tự và **xóa** chúng theo thứ tự chèn **ngược**, thì lúc xóa `x`, hai hàng xóm còn lại của `x` trong danh sách chính là tiền nhiệm/hậu bối **lúc chèn `x`** (vì các phần tử chèn sau `x` đã bị xóa trước).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — `SortedList` + độ sâu theo tiền nhiệm/hậu bối

In [8]:
def max_depth_sortedlist(order):
    sl = SortedList()
    depth = {}
    best = 0
    for x in order:
        i = sl.bisect_left(x)
        dp = depth[sl[i - 1]] if i > 0 else 0
        ds = depth[sl[i]] if i < len(sl) else 0
        depth[x] = 1 + max(dp, ds)
        best = max(best, depth[x])
        sl.add(x)
    return best

for order, want in examples:
    print(max_depth_sortedlist(order), "kỳ vọng", want)

3 kỳ vọng 3
3 kỳ vọng 3
4 kỳ vọng 4


Code nộp LeetCode của Cách 2:

```python
from sortedcontainers import SortedList

class Solution:
    def maxDepthBST(self, order: List[int]) -> int:
        sl = SortedList()
        depth = [0] * (len(order) + 2)
        best = 0
        for x in order:
            i = sl.bisect_left(x)
            d = 1 + max(depth[sl[i - 1]] if i else 0, depth[sl[i]] if i < len(sl) else 0)
            depth[x] = d
            best = max(best, d)
            sl.add(x)
        return best
```

**Độ phức tạp:** Time `O(n log n)`, Space `O(n)`.

### 7.2 Cách 3 — Danh sách liên kết đôi, xóa theo thứ tự ngược (`O(n)`)

Từng đoạn:
1. Danh sách `0 ↔ 1 ↔ 2 ↔ … ↔ n ↔ n+1` (0 và `n+1` là lính canh).
2. Duyệt `order` **ngược**: với `x`, ghi `pred[x] = prv[x]`, `succ[x] = nxt[x]`, rồi xóa `x`.
3. Duyệt `order` **xuôi**: `depth[x] = 1 + max(depth[pred[x]], depth[succ[x]])` (lính canh có độ sâu 0).

In [9]:
def max_depth_linked(order):
    n = len(order)
    prv = list(range(-1, n + 1))          # prv[v] = v - 1
    nxt = list(range(1, n + 3))           # nxt[v] = v + 1
    pred = [0] * (n + 2)
    succ = [0] * (n + 2)
    for x in reversed(order):
        pred[x], succ[x] = prv[x], nxt[x]
        nxt[prv[x]] = nxt[x]
        prv[nxt[x]] = prv[x]
    depth = [0] * (n + 2)                 # depth[0] = depth[n+1] = 0 (lính canh)
    best = 0
    for x in order:
        d = 1 + max(depth[pred[x]], depth[succ[x]])
        depth[x] = d
        if d > best:
            best = d
    return best

for order, want in examples:
    print(max_depth_linked(order), "kỳ vọng", want)

3 kỳ vọng 3
3 kỳ vọng 3
4 kỳ vọng 4


**Độ phức tạp:** Time `O(n)`, Space `O(n)`. Không cần cấu trúc dữ liệu phức tạp.

### 7.3 Cách 4 — Cây Cartesian bằng ngăn xếp đơn điệu (`O(n)`)

Gán `time[v]` = vị trí của `v` trong `order`. BST theo thứ tự chèn là **cây Cartesian** của dãy `time[1], time[2], …, time[n]` (duyệt trung thứ tự = giá trị tăng dần, thuộc tính heap = thời điểm chèn nhỏ ở trên). Dựng bằng ngăn xếp tăng dần theo `time`, rồi tính độ sâu bằng BFS/DFS lặp.

In [10]:
def max_depth_cartesian(order):
    n = len(order)
    time_ = [0] * (n + 1)
    for t, v in enumerate(order):
        time_[v] = t
    left = [0] * (n + 1); right = [0] * (n + 1)    # 0 = không có con
    st = []
    for v in range(1, n + 1):                      # duyệt theo giá trị tăng dần
        last = 0
        while st and time_[st[-1]] > time_[v]:     # các nút chèn SAU v phải xuống dưới v
            last = st.pop()
        left[v] = last
        if st:
            right[st[-1]] = v
        st.append(v)
    rootv = st[0]
    best = 0
    stack = [(rootv, 1)]
    while stack:
        v, d = stack.pop()
        if d > best: best = d
        if left[v]: stack.append((left[v], d + 1))
        if right[v]: stack.append((right[v], d + 1))
    return best

for order, want in examples:
    print(max_depth_cartesian(order), "kỳ vọng", want)

3 kỳ vọng 3
3 kỳ vọng 3
4 kỳ vọng 4


**Độ phức tạp:** mỗi giá trị vào/ra ngăn xếp một lần ⇒ dựng `O(n)`; DFS `O(n)` ⇒ Time `O(n)`, Space `O(n)`.

## 8. TRACE từng bước

Trace Cách 2 trên `order = [3, 1, 4, 2, 5]`:

In [11]:
def trace_sortedlist(order):
    sl = SortedList(); depth = {}
    for x in order:
        i = sl.bisect_left(x)
        p = sl[i - 1] if i > 0 else None; s = sl[i] if i < len(sl) else None
        dp = depth.get(p, 0); ds = depth.get(s, 0)
        depth[x] = 1 + max(dp, ds); sl.add(x)
        parent = p if dp >= ds and p is not None else s
        print(f"chèn {x}: tiền nhiệm {p} (sâu {dp}), hậu bối {s} (sâu {ds}) -> gắn vào {parent}, độ sâu {depth[x]}")
    print("độ sâu cây:", max(depth.values()))

trace_sortedlist([3, 1, 4, 2, 5])

chèn 3: tiền nhiệm None (sâu 0), hậu bối None (sâu 0) -> gắn vào None, độ sâu 1
chèn 1: tiền nhiệm None (sâu 0), hậu bối 3 (sâu 1) -> gắn vào 3, độ sâu 2
chèn 4: tiền nhiệm 3 (sâu 1), hậu bối None (sâu 0) -> gắn vào 3, độ sâu 2
chèn 2: tiền nhiệm 1 (sâu 2), hậu bối 3 (sâu 1) -> gắn vào 1, độ sâu 3
chèn 5: tiền nhiệm 4 (sâu 2), hậu bối None (sâu 0) -> gắn vào 4, độ sâu 3
độ sâu cây: 3


In [12]:
def trace_linked(order):
    n = len(order); prv = list(range(-1, n + 1)); nxt = list(range(1, n + 3))
    pred = [0] * (n + 2); succ = [0] * (n + 2)
    print("xóa theo thứ tự ngược:")
    for x in reversed(order):
        pred[x], succ[x] = prv[x], nxt[x]
        print(f"  xóa {x}: hàng xóm hiện tại = ({prv[x]}, {nxt[x]}) -> tiền nhiệm/hậu bối lúc chèn {x}")
        nxt[prv[x]] = nxt[x]; prv[nxt[x]] = prv[x]
    depth = [0] * (n + 2)
    print("tính độ sâu theo thứ tự chèn (0 và n+1 là lính canh):")
    for x in order:
        depth[x] = 1 + max(depth[pred[x]], depth[succ[x]])
        print(f"  {x}: 1 + max(depth[{pred[x]}]={depth[pred[x]]}, depth[{succ[x]}]={depth[succ[x]]}) = {depth[x]}")

trace_linked([3, 1, 4, 2, 5])

xóa theo thứ tự ngược:
  xóa 5: hàng xóm hiện tại = (4, 6) -> tiền nhiệm/hậu bối lúc chèn 5
  xóa 2: hàng xóm hiện tại = (1, 3) -> tiền nhiệm/hậu bối lúc chèn 2
  xóa 4: hàng xóm hiện tại = (3, 6) -> tiền nhiệm/hậu bối lúc chèn 4
  xóa 1: hàng xóm hiện tại = (0, 3) -> tiền nhiệm/hậu bối lúc chèn 1
  xóa 3: hàng xóm hiện tại = (0, 6) -> tiền nhiệm/hậu bối lúc chèn 3
tính độ sâu theo thứ tự chèn (0 và n+1 là lính canh):
  3: 1 + max(depth[0]=0, depth[6]=0) = 1
  1: 1 + max(depth[0]=0, depth[3]=1) = 2
  4: 1 + max(depth[3]=1, depth[6]=0) = 2
  2: 1 + max(depth[1]=2, depth[3]=1) = 3
  5: 1 + max(depth[4]=2, depth[6]=0) = 3


In [13]:
root = None
for x in [3, 1, 4, 2, 5]:
    root, _ = insert(root, x)
print("cây thật (level-order):", to_level_list(root), "| độ sâu:", depth_iter(root))

cây thật (level-order): [3, 1, 4, None, 2, None, 5] | độ sâu: 3


## 9. Code nộp LeetCode

Chọn Cách 3 (`O(n)`, không phụ thuộc thư viện ngoài).

In [14]:
class Solution:
    def maxDepthBST(self, order: List[int]) -> int:
        n = len(order)
        prv = list(range(-1, n + 1))
        nxt = list(range(1, n + 3))
        pred = [0] * (n + 2)
        succ = [0] * (n + 2)
        for x in reversed(order):
            pred[x], succ[x] = prv[x], nxt[x]
            nxt[prv[x]] = nxt[x]
            prv[nxt[x]] = prv[x]
        depth = [0] * (n + 2)
        best = 0
        for x in order:
            d = 1 + max(depth[pred[x]], depth[succ[x]])
            depth[x] = d
            if d > best:
                best = d
        return best

sol = Solution()
for order, want in examples:
    got = sol.maxDepthBST(order)
    print(got, "OK" if got == want else "SAI")

3 OK
3 OK
4 OK


## 10. Vì sao thuật toán đúng

**Bổ đề (tiền nhiệm/hậu bối có quan hệ tổ tiên).** Giả sử `p < s` là hai giá trị kề nhau trong tập `S` đã chèn (không có giá trị nào của `S` nằm giữa). Gọi `r` là tổ tiên chung thấp nhất của `p` và `s` trong BST. Nếu `r ∉ {p, s}` thì `p` ở cây con trái của `r`, `s` ở cây con phải ⇒ `p < r < s`, mâu thuẫn. Vậy `r = p` hoặc `r = s`.

**Bổ đề (vị trí gắn).** Với `x` mới chèn có `p < x < s`:
- Nếu `p` là tổ tiên của `s`: `s` ở cây con phải của `p`, và `s` là giá trị **nhỏ nhất** trong cây con đó (vì không có giá trị nào giữa `p` và `s`) ⇒ `s.left` trống. Đường chèn `x` đi qua `p` (rẽ phải vì `x > p`), rồi trong cây con phải của `p` nó luôn so sánh với các nút `≥ s` (cây con này không chứa giá trị nào trong `(p, s)`), nên rẽ trái cho tới khi gặp `s` và dừng ở `s.left` (trống). ⇒ `x` là con trái của `s` — nút sâu hơn.
- Đối xứng nếu `s` là tổ tiên của `p`.
- Chỉ có một trong hai (tồn tại `p` hoặc `s`) thì gắn vào nút đó.

⇒ `depth(x) = 1 + max(depth(p), depth(s))` (nút sâu hơn là con cháu của nút kia).

**Xóa ngược.** Khi xóa `x` (theo thứ tự ngược), các phần tử còn trong danh sách là đúng các phần tử chèn **trước** `x` (cùng `x`) cộng lính canh ⇒ hàng xóm của `x` là tiền nhiệm/hậu bối lúc chèn `x`. ✓

**Cartesian.** Cây Cartesian của dãy `time[1..n]` là cây duy nhất có trung thứ tự `1..n` và tính chất heap theo `time`; BST chèn theo `order` cũng thỏa hai tính chất đó (cha luôn được chèn trước con) ⇒ hai cây trùng nhau.

## 11. Phân tích Time / Space

Gọi `n ≤ 10⁵`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Chèn BST thật | `O(n · h)`, worst `O(n²)` | `O(n)` | thứ tự tăng dần rất chậm |
| 2. `SortedList` | `O(n log n)` | `O(n)` | trực tuyến |
| 3. Liên kết đôi + xóa ngược | `O(n)` | `O(n)` | **nộp bài** |
| 4. Cartesian + ngăn xếp | `O(n)` | `O(n)` | dựng được cả cây |

In [15]:
rng = random.Random(1)
n = 10**5
cases = {"ngẫu nhiên": rng.sample(range(1, n + 1), n), "tăng dần": list(range(1, n + 1)),
         "zigzag": [x for pair in zip(range(1, n // 2 + 1), range(n, n // 2, -1)) for x in pair]}
for name, order in cases.items():
    line = f"{name:10s}:"
    res = []
    for fname, fn in [("sortedlist", max_depth_sortedlist), ("linked", max_depth_linked), ("cartesian", max_depth_cartesian),
                      ("submit", Solution().maxDepthBST)]:
        t0 = time.perf_counter(); res.append(fn(order)); line += f" {fname}={time.perf_counter() - t0:.3f}s"
    assert len(set(res)) == 1
    print(line, "-> độ sâu", res[0])

ngẫu nhiên: sortedlist=0.324s linked=0.051s cartesian=0.027s submit=0.049s -> độ sâu 40
tăng dần  : sortedlist=0.152s linked=0.017s cartesian=0.017s submit=0.016s -> độ sâu 100000


zigzag    : sortedlist=0.304s linked=0.016s cartesian=0.020s submit=0.017s -> độ sâu 100000


Với hoán vị ngẫu nhiên, độ sâu BST kỳ vọng khoảng `4.3 ln n ≈ 50` (kết quả thực nghiệm in ở trên); thứ tự tăng dần và "zigzag" cho cây suy biến độ sâu `n`.

## 12. Bộ test

In [16]:
fns = [max_depth_sortedlist, max_depth_linked, max_depth_cartesian, lambda o: Solution().maxDepthBST(o)]

for order, want in examples:
    assert max_depth_insert(order) == want
    for fn in fns:
        assert fn(order) == want

edge = [([1], 1), ([1, 2], 2), ([2, 1], 2), ([2, 1, 3], 2), ([3, 2, 1], 3), ([4, 2, 6, 1, 3, 5, 7], 3),
        ([1, 5, 2, 4, 3], 5)]
for order, want in edge:
    assert max_depth_insert(order) == want, order
    for fn in fns:
        assert fn(order) == want, (fn, order)

rng = random.Random(2024)
for _ in range(1500):
    n = rng.randint(1, 60)
    order = rng.sample(range(1, n + 1), n)
    want = max_depth_insert(order)
    for fn in fns:
        assert fn(order) == want, (fn, order)
print("ví dụ + edge + 1500 hoán vị ngẫu nhiên khớp chèn BST thật")

ví dụ + edge + 1500 hoán vị ngẫu nhiên khớp chèn BST thật


In [17]:
# kiểm tra mạnh hơn: độ sâu TỪNG nút (không chỉ độ sâu cây) từ công thức khớp cây thật
def depths_formula(order):
    n = len(order); prv = list(range(-1, n + 1)); nxt = list(range(1, n + 3))
    pred = [0] * (n + 2); succ = [0] * (n + 2)
    for x in reversed(order):
        pred[x], succ[x] = prv[x], nxt[x]; nxt[prv[x]] = nxt[x]; prv[nxt[x]] = prv[x]
    depth = [0] * (n + 2)
    for x in order:
        depth[x] = 1 + max(depth[pred[x]], depth[succ[x]])
    return depth[1:n + 1]

def depths_real(order):
    root = None; d = {}
    for x in order:
        root, dx = insert(root, x); d[x] = dx
    return [d[v] for v in range(1, len(order) + 1)]

rng = random.Random(9)
for _ in range(300):
    n = rng.randint(1, 80); order = rng.sample(range(1, n + 1), n)
    assert depths_formula(order) == depths_real(order)
print("độ sâu từng nút theo công thức tiền nhiệm/hậu bối khớp cây thật (300 hoán vị)")

độ sâu từng nút theo công thức tiền nhiệm/hậu bối khớp cây thật (300 hoán vị)


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Chèn BST bằng **đệ quy** trên cây suy biến (`RecursionError`)

In [18]:
import sys

def insert_rec(node, x):
    if node is None:
        return TreeNode(x)
    if x < node.val:
        node.left = insert_rec(node.left, x)
    else:
        node.right = insert_rec(node.right, x)
    return node

root = None
try:
    for x in range(1, 3000):
        root = insert_rec(root, x)
    print("chạy được")
except RecursionError as e:
    print("RecursionError:", str(e)[:50], f"(giới hạn {sys.getrecursionlimit()})")

RecursionError: maximum recursion depth exceeded (giới hạn 1000)


**Sửa:** chèn lặp, hoặc tốt hơn — không dựng cây (Cách 2–4).

### Lỗi 2 — Luôn gắn vào tiền nhiệm (hoặc luôn hậu bối)

In [19]:
def wrong_always_pred(order):
    sl = SortedList(); depth = {}
    for x in order:
        i = sl.bisect_left(x)
        if i > 0: depth[x] = depth[sl[i - 1]] + 1        # SAI: phải chọn nút sâu hơn trong (pred, succ)
        elif i < len(sl): depth[x] = depth[sl[i]] + 1
        else: depth[x] = 1
        sl.add(x)
    return max(depth.values())

for order, want in examples:
    print("sai =", wrong_always_pred(order), "| đúng =", want)

sai = 2 | đúng = 3
sai = 3 | đúng = 3
sai = 4 | đúng = 4


Ví dụ 1: chèn `3` có tiền nhiệm `2` (sâu 1) và hậu bối `4` (sâu 2) — gắn vào `4`, không phải `2`. **Sửa:** `1 + max(depth[pred], depth[succ])`.

### Lỗi 3 — Xóa xuôi thay vì ngược (lấy hàng xóm trong tập **cuối cùng**)

In [20]:
def wrong_forward_neighbors(order):
    n = len(order); depth = [0] * (n + 2)
    for x in order:
        depth[x] = 1 + max(depth[x - 1] if x > 1 else 0, depth[x + 1] if x < n else 0)   # SAI: x-1, x+1 có thể chưa chèn
    return max(depth)

for order, want in examples:
    print("sai =", wrong_forward_neighbors(order), "| đúng =", want)

sai = 2 | đúng = 3
sai = 3 | đúng = 3
sai = 4 | đúng = 4


Hàng xóm theo **giá trị** `x ± 1` chưa chắc đã được chèn trước `x` (độ sâu 0 hoặc… lấy nhầm độ sâu của nút chèn sau). **Sửa:** tiền nhiệm/hậu bối trong tập **đã chèn** — xóa ngược để có chúng.

### Lỗi 4 — Độ sâu tính theo **cạnh** thay vì **nút**

In [21]:
def wrong_edges(order):
    return Solution().maxDepthBST(order) - 1              # SAI: đề đếm số NÚT trên đường

for order, want in examples:
    print("sai =", wrong_edges(order), "| đúng =", want)

sai = 2 | đúng = 3
sai = 2 | đúng = 3
sai = 3 | đúng = 4


**Sửa:** gốc có độ sâu 1 (đề: "number of nodes along the longest path").

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chèn tuần tự vào BST, hỏi độ sâu/cha" ⇒ **tiền nhiệm/hậu bối** trong tập đã chèn.
- Cần tiền nhiệm/hậu bối trực tuyến ⇒ `SortedList` / TreeSet; ngoại tuyến với giá trị `1..n` ⇒ **xóa ngược trên danh sách liên kết**.
- BST theo thứ tự chèn = **cây Cartesian** (khóa = giá trị, ưu tiên = thời gian) — dựng bằng ngăn xếp đơn điệu.
- Tránh đệ quy trên cây có thể suy biến (`n = 10⁵`).
- Họ bài: 1008 *Construct BST from Preorder*, 654 *Maximum Binary Tree* (Cartesian), 84 *Largest Rectangle in Histogram* (ngăn xếp đơn điệu), 1649 *Create Sorted Array through Instructions*.

## 15. Biến thể của bài

1. **Trả về cha của mỗi nút** khi chèn: nút sâu hơn trong (tiền nhiệm, hậu bối).
2. **Giá trị tùy ý (không phải hoán vị `1..n`)**: nén tọa độ rồi dùng Cách 3, hoặc dùng `SortedList`.
3. **Truy vấn trực tuyến** (phải trả lời độ sâu ngay sau mỗi lần chèn): Cách 2.
4. **Tổng độ sâu mọi nút** (chi phí xây cây): cộng `depth[x]`.

In [22]:
def parents(order):
    """Biến thể 1: cha của mỗi nút (0 nếu là gốc)."""
    sl = SortedList(); depth = {}; par = {}
    for x in order:
        i = sl.bisect_left(x)
        p = sl[i - 1] if i > 0 else None; s = sl[i] if i < len(sl) else None
        dp, ds = depth.get(p, 0), depth.get(s, 0)
        par[x] = (p if dp > ds else s) if (p is not None or s is not None) else 0
        depth[x] = 1 + max(dp, ds); sl.add(x)
    return par

def parents_real(order):
    root = None; par = {}
    for x in order:
        if root is None:
            root = TreeNode(x); par[x] = 0; continue
        cur = root
        while True:
            nxt_ = cur.left if x < cur.val else cur.right
            if nxt_ is None:
                if x < cur.val: cur.left = TreeNode(x)
                else: cur.right = TreeNode(x)
                par[x] = cur.val; break
            cur = nxt_
    return par

def max_depth_any_values(values):
    """Biến thể 2: giá trị phân biệt tùy ý -> nén tọa độ."""
    rank = {v: i + 1 for i, v in enumerate(sorted(values))}
    return Solution().maxDepthBST([rank[v] for v in values])

rng = random.Random(3)
for _ in range(300):
    n = rng.randint(1, 40); order = rng.sample(range(1, n + 1), n)
    assert parents(order) == parents_real(order)
print("cha của mỗi nút khớp cây thật;", "giá trị tùy ý:", max_depth_any_values([50, 10, 70, 60, 65]))

cha của mỗi nút khớp cây thật; giá trị tùy ý: 4


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Chèn BST | mô phỏng | `O(n²)` worst | `O(n)` | kiểm chứng |
| 2. `SortedList` | `1 + max(d(pred), d(succ))` | `O(n log n)` | `O(n)` | trực tuyến |
| 3. Xóa ngược | liên kết đôi | `O(n)` | `O(n)` | **nộp bài** |
| 4. Cartesian | ngăn xếp đơn điệu | `O(n)` | `O(n)` | cần cấu trúc cây |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `maxDepthBST([3, 2, 1, 4, 5])` và `maxDepthBST([4, 2, 6, 1, 3, 5, 7])`?

**Bài 2 (tính tay):** với `order = [4, 1, 6, 5, 2, 7, 3]` lập bảng (chèn, tiền nhiệm, hậu bối, độ sâu).

**Bài 3 (điền TODO):** hoàn thiện `insertion_depths(order)` trả về mảng độ sâu theo **thứ tự chèn** (dùng Cách 3).

**Bài 4 (tự viết lại):** viết lại Cách 2 **không** dùng `sortedcontainers` mà dùng `bisect.insort` trên list và đo thời gian với `n = 10⁵` (thứ tự ngẫu nhiên).

**Bài 5 (mini challenge):** tìm một thứ tự chèn của `1..n` cho BST có độ sâu **nhỏ nhất** (`⌈log₂(n+1)⌉`) — gợi ý: chèn trung vị trước (BFS trên các đoạn). Kiểm chứng độ sâu bằng `maxDepthBST`.

In [23]:
def insertion_depths(order):
    # TODO: xóa ngược lấy pred/succ, rồi depth[x] = 1 + max(...) theo thứ tự chèn
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) `[3,2,1,4,5]`: nhánh trái `3→2→1`, nhánh phải `3→4→5`. (2) Đều dài 3. (3) Đáp án `3`; cây cân bằng 7 nút ⇒ `3`.

**Bài 2:** (1) `4` gốc (1); `1`: hậu bối 4 ⇒ 2; `6`: tiền nhiệm 4 ⇒ 2. (2) `5`: (4, 6) sâu (1, 2) ⇒ 3; `2`: (1, 4) sâu (2, 1) ⇒ 3; `7`: tiền nhiệm 6 ⇒ 3. (3) `3`: (2, 4) sâu (3, 1) ⇒ 4 ⇒ độ sâu cây `4`.

**Bài 3:** (1) Copy Cách 3. (2) Thay vì `best`, lưu `out.append(d)`. (3) Kiểm tra `max(out)` bằng đáp án.

**Bài 4:** (1) `insort(lst, x)` là `O(n)` mỗi lần (dịch mảng). (2) Tìm `pred/succ` bằng `bisect_left`. (3) So thời gian với `SortedList`.

**Bài 5:** (1) Hàng đợi các đoạn `[l, r]`. (2) Lấy `mid = (l + r) // 2`, thêm vào thứ tự, đẩy `[l, mid−1]`, `[mid+1, r]`. (3) Độ sâu `= n.bit_length()`.

## 19. Đáp án tham khảo

In [24]:
# Bài 1
assert Solution().maxDepthBST([3, 2, 1, 4, 5]) == 3 and Solution().maxDepthBST([4, 2, 6, 1, 3, 5, 7]) == 3

# Bài 2
trace_sortedlist([4, 1, 6, 5, 2, 7, 3])
assert Solution().maxDepthBST([4, 1, 6, 5, 2, 7, 3]) == 4

# Bài 3
def insertion_depths_ans(order):
    d = depths_formula(order)              # d[v-1] = độ sâu của giá trị v
    return [d[x - 1] for x in order]
assert insertion_depths_ans([2, 1, 4, 3]) == [1, 2, 2, 3]

# Bài 4
def max_depth_insort(order):
    lst = []; depth = {}; best = 0
    for x in order:
        i = bisect_left(lst, x)
        d = 1 + max(depth[lst[i - 1]] if i else 0, depth[lst[i]] if i < len(lst) else 0)
        depth[x] = d; best = max(best, d); insort(lst, x)
    return best

rng = random.Random(261)
order = rng.sample(range(1, 10**5 + 1), 10**5)
t0 = time.perf_counter(); a = max_depth_insort(order); t1 = time.perf_counter(); b = max_depth_sortedlist(order); t2 = time.perf_counter()
assert a == b
print(f"Bài 4: insort {t1 - t0:.3f}s | SortedList {t2 - t1:.3f}s | độ sâu {a}")
print("Bài 1-4: OK")

chèn 4: tiền nhiệm None (sâu 0), hậu bối None (sâu 0) -> gắn vào None, độ sâu 1
chèn 1: tiền nhiệm None (sâu 0), hậu bối 4 (sâu 1) -> gắn vào 4, độ sâu 2
chèn 6: tiền nhiệm 4 (sâu 1), hậu bối None (sâu 0) -> gắn vào 4, độ sâu 2
chèn 5: tiền nhiệm 4 (sâu 1), hậu bối 6 (sâu 2) -> gắn vào 6, độ sâu 3
chèn 2: tiền nhiệm 1 (sâu 2), hậu bối 4 (sâu 1) -> gắn vào 1, độ sâu 3
chèn 7: tiền nhiệm 6 (sâu 2), hậu bối None (sâu 0) -> gắn vào 6, độ sâu 3
chèn 3: tiền nhiệm 2 (sâu 3), hậu bối 4 (sâu 1) -> gắn vào 2, độ sâu 4
độ sâu cây: 4


Bài 4: insort 0.442s | SortedList 0.307s | độ sâu 40
Bài 1-4: OK


In [25]:
# Bài 5 — thứ tự chèn cho cây độ sâu nhỏ nhất
def balanced_order(n):
    out = []; q = deque([(1, n)])
    while q:
        l, r = q.popleft()
        if l > r: continue
        mid = (l + r) // 2; out.append(mid)
        q.append((l, mid - 1)); q.append((mid + 1, r))
    return out

for n in [1, 2, 3, 7, 8, 100, 1000, 10**5]:
    o = balanced_order(n)
    assert sorted(o) == list(range(1, n + 1))
    assert Solution().maxDepthBST(o) == n.bit_length()
print("Bài 5: độ sâu nhỏ nhất = n.bit_length() =", [n.bit_length() for n in [1, 2, 3, 7, 8, 100, 1000, 10**5]])

Bài 5: độ sâu nhỏ nhất = n.bit_length() = [1, 2, 2, 3, 4, 7, 10, 17]


## 20. Template nhớ nhanh + Checklist

```python
# Độ sâu BST theo thứ tự chèn (order là hoán vị 1..n): xóa ngược trên danh sách liên kết
prv = list(range(-1, n + 1)); nxt = list(range(1, n + 3))       # 0 và n+1 là lính canh
for x in reversed(order):                                        # hàng xóm lúc xóa = pred/succ lúc chèn
    pred[x], succ[x] = prv[x], nxt[x]
    nxt[prv[x]] = nxt[x]; prv[nxt[x]] = prv[x]
for x in order:
    depth[x] = 1 + max(depth[pred[x]], depth[succ[x]])          # gắn vào nút SÂU hơn
return max(depth)
```

**Checklist trước khi submit:**
- [ ] Độ sâu đếm theo **nút** (gốc = 1).
- [ ] Gắn vào **nút sâu hơn** trong (tiền nhiệm, hậu bối).
- [ ] Tiền nhiệm/hậu bối trong tập **đã chèn** (xóa ngược / SortedList), không phải `x ± 1`.
- [ ] Lính canh có độ sâu 0.
- [ ] Không đệ quy trên cây có thể sâu `10⁵`.